### Tutorial Walks in BuilDyn

BuilDyn makes system identification via Pseudo Random walks fairly easy.   
In ithis notebooke we will go through already pre-implemented walks, how to use them on the FMU object, and how to create custom random walks. We start by initializing a FMU object.

In [ ]:
from buildyn import FMU

# Example FMU from BuilDa 2.0
fmu_path = "resources/building.fmu"

# Initial variables from the BuilDa 2.0 FMU - without those the FMU cannot be initialized.
start_variables = {
    "weaDat.filNam": "resources/Munich.mos",
    "internalGain.fileName": "resources/NoActivity.txt",
    "hygienicalWindowOpening.fileName": "resources/no_opening.txt",
    "UseInternalController.k": 0
}

# FMU object from the buildyn package
fmu = FMU(fmu_file=fmu_path, init_values=start_variables)

In this case, the FMU would always have no heating. This would be constantly the same for the whole simulation every time.

In [ ]:
observables = ["thermalZone.TAir", "ctrSignalHeating"]

# We can simulate the FMU simply by calling the simulate method.
one_day_df = fmu.simulate(observables=observables)

# We see a constant line here.
one_day_df["ctrSignalHeating"].plot()

To explore the space with different heatings, we can now define walks to "walk" over different settings. For instance:

In [ ]:
from buildyn.walker.fixed.poisson_walker import PoissonWalker
from buildyn.walker.interval_walker import IntervalWalker

# First we again copy the inintial FMU
fmu = fmu.__copy__()

# We initialize a (pseudo-) random walker that changes the heating control (between 0 and 1) in the FMU while simulating.
random_walker = PoissonWalker(lam=8, is_discrete=False)

# We then set a interval in which the random_walker will change the heating contols. 
# For instance, in the following line we set it to 900, meaning the heating signal wil change every 15 minutes.
interval_walker = IntervalWalker(walker=random_walker, interval=900)

# As we can apply multiple walker to one FMU, and the walker needs to be applied to one parameter, we write it in a dictionary.
walkers = {
    "ctrSignalHeating": interval_walker
}

# Now we can simulate the FMU with the interval_walker in action.
df_walker = fmu.simulate(observables=observables, walker=walkers)

df_walker["ctrSignalHeating"].plot()

This walker clearly creates another output than the original FMU without any walkers implemented.  
We can also compare the indoor temperatures with and without walker.

In [ ]:
from matplotlib import pyplot as plt

plt.figure(figsize=(12, 4))

plt.plot(one_day_df.index, one_day_df["thermalZone.TAir"], label="No Walker")
plt.plot(df_walker.index, df_walker["thermalZone.TAir"], label="With Walker")

plt.ylabel("Heating Signal")

plt.legend()
plt.show()

If the walker above do not match expectations, we can also create our own walkers with MoPrior. For instance, we want a walker that sets heating on for 10 steps and then off for 10 and so on.

In [ ]:
from buildyn.walker.walker import Walker

# Our custom Walker class we can use in combination with the MoPrior package.
class CustomWalker(Walker):

    def __init__(self, n: int = 1):
        self.n = n

    # Override this walk function.
    def walk_pattern(self, num_steps: int = 1):

        result = []
        val = 0
        while len(result) < num_steps:
            result.extend([val] * self.n)
            val = 1 - val   # 1 to 0 and the other way around

        return result[:num_steps]


# Apply the custom walker
fmu = fmu.__copy__()

custom_walker = CustomWalker(n=10)
custom_iw = IntervalWalker(walker=custom_walker, interval=900)

walkers = {
    "ctrSignalHeating": custom_iw
}

df_custom_walker = fmu.simulate(observables=observables, walker=walkers)

df_custom_walker["thermalZone.TAir"].plot()